# Government720

## Training of Federated Dataset

### Importing and Splitting Data

In [1]:
# Access Google Drive for Excel file
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Import packages
import pandas as pd
import tensorflow as tf
import numpy as np

In [3]:
# Read in federated client datasets
file_path = '/content/drive/My Drive/G720_FEDERATED.xlsx'  # Finalized dataset
sheets = pd.ExcelFile(file_path).sheet_names

# Define split date
split_date = '2021-01-01'

client_data = {}  # Dictionary to store client datasets
# Open Excel sheet and split data into training and testing per client
for sheet in sheets:
    df = pd.read_excel(file_path, sheet_name=sheet)
    # Define training and testing sets based on date split
    train_mask = df["DATE"] < split_date
    test_mask = df["DATE"] >= split_date
    train_df = df[train_mask]
    test_df = df[test_mask]
    # Set DATE to index
    train_df.set_index("DATE", inplace=True)
    test_df.set_index("DATE", inplace=True)
    # Define X_train, y_train, X_test, y_test for each client
    labels_train = train_df['GOVT_SATISFACTION'].values
    features_train = train_df.drop('GOVT_SATISFACTION', axis=1).values
    X_train = features_train
    y_train = labels_train
    labels_test = test_df['GOVT_SATISFACTION'].values
    features_test = test_df.drop('GOVT_SATISFACTION', axis=1).values
    X_test = features_test
    y_test = labels_test
    client_data[sheet] = {'X_train': X_train, 'y_train': y_train, 'X_test': X_test, 'y_test': y_test}

In [4]:
print(client_data)

{'CLIENT1': {'X_train': array([[ 9.07713402e-02,  4.28571429e-01,  0.00000000e+00, ...,
         6.58336037e-02,  0.00000000e+00,  5.63253420e+00],
       [ 8.83358744e-02,  4.39460188e-01, -6.45062393e-03, ...,
         7.15544939e-02,  3.96384768e-03,  5.60521660e+00],
       [ 6.83087639e-02,  4.61813803e-01,  2.08505325e-02, ...,
         7.07639889e-02, -5.04907004e-03,  5.60910131e+00],
       ...,
       [ 2.71868036e-01,  3.14350154e-01,  6.35025506e-01, ...,
         4.38141108e-02,  7.32617687e-01,  3.54299721e-01],
       [ 2.70880152e-01,  3.27219170e-01,  6.66045050e-01, ...,
         3.32215925e-02,  7.23261049e-01,  3.54220929e-01],
       [ 2.58172910e-01,  3.23932540e-01,  6.68965388e-01, ...,
         2.82055433e-02,  7.26318093e-01,  3.56262531e-01]]), 'y_train': array([0.98412698, 0.96993573, 0.97818637, ..., 0.09899512, 0.07237834,
       0.06976402]), 'X_test': array([[0.26315789, 0.3125    , 0.64705882, ..., 0.02469136, 0.72293987,
        0.35290557],
       [0.

### FL Model Setup

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.optimizers import Adam

In [6]:
def create_client_model(input_dim, embedding_dim=8):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, activation='relu'),
        Dense(embedding_dim, activation='relu')  # Output an embedding
    ])
    return model

In [7]:
def create_server_model(num_clients, embedding_dim=8):
    total_input_dim = num_clients * embedding_dim
    model = Sequential([
        Input(shape=(total_input_dim,)),
        Dense(32, activation='relu'),
        Dense(1)  # Regression output
    ])
    return model

In [8]:
# Parameters
embedding_dim = 8
epochs = 5
batch_size = 32
learning_rate = 0.001

In [9]:
# Create client models
client_models = {}
for client_id, data in client_data.items():
    input_dim = data['X_train'].shape[1]
    model = create_client_model(input_dim, embedding_dim)
    client_models[client_id] = model
    print(f"Client {client_id} model created with input dim {input_dim}")

Client CLIENT1 model created with input dim 29
Client CLIENT2 model created with input dim 6
Client CLIENT3 model created with input dim 20
Client CLIENT4 model created with input dim 10
Client CLIENT5 model created with input dim 6
Client CLIENT6 model created with input dim 12
Client CLIENT7 model created with input dim 6
Client CLIENT8 model created with input dim 10
Client CLIENT9 model created with input dim 4
Client CLIENT10 model created with input dim 2
Client CLIENT11 model created with input dim 16
Client CLIENT12 model created with input dim 3


In [10]:
num_clients = len(client_models)
server_model = create_server_model(num_clients, embedding_dim)
print("Server model created.")

Server model created.


In [13]:
# Separate optimizers
client_optimizers = {client_id: Adam(learning_rate) for client_id in client_models}
server_optimizer = Adam(learning_rate)

### FL Model Training

In [14]:
# Assume all clients have the same number of samples (1092)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    for i in range(0, 1092, batch_size):
        # Collect embeddings from all clients
        client_embeddings = []

        with tf.GradientTape(persistent=True) as tape:
            for client_id, model in client_models.items():
                X_batch = client_data[client_id]['X_train'][i:i+batch_size]
                embedding = model(X_batch, training=True)
                client_embeddings.append(embedding)

            # Concatenate all client embeddings
            combined_embedding = tf.concat(client_embeddings, axis=1)

            # Get true labels (only need from one client)
            y_true = client_data[list(client_models.keys())[0]]['y_train'][i:i+batch_size]

            # Server prediction
            y_pred = server_model(combined_embedding, training=True)

            # Loss
            loss = tf.reduce_mean(tf.square(y_true - tf.squeeze(y_pred)))

        # Compute gradients
        server_grads = tape.gradient(loss, server_model.trainable_variables)
        server_optimizer.apply_gradients(zip(server_grads, server_model.trainable_variables))

        for client_id, model in client_models.items():
            client_grads = tape.gradient(loss, model.trainable_variables)
            client_optimizers[client_id].apply_gradients(zip(client_grads, model.trainable_variables))

        print(f"Batch {i//batch_size + 1}: Loss = {loss.numpy():.4f}")


Epoch 1/5
Batch 1: Loss = 5.0448
Batch 2: Loss = 3.0890
Batch 3: Loss = 1.6801
Batch 4: Loss = 1.0830
Batch 5: Loss = 0.2561
Batch 6: Loss = 0.0976
Batch 7: Loss = 0.0057
Batch 8: Loss = 0.0209
Batch 9: Loss = 0.0392
Batch 10: Loss = 0.1243
Batch 11: Loss = 0.1731
Batch 12: Loss = 0.1103
Batch 13: Loss = 0.1294
Batch 14: Loss = 0.1658
Batch 15: Loss = 0.1252
Batch 16: Loss = 0.0096
Batch 17: Loss = 0.0370
Batch 18: Loss = 0.0312
Batch 19: Loss = 0.0301
Batch 20: Loss = 0.0156
Batch 21: Loss = 0.0191
Batch 22: Loss = 0.0146
Batch 23: Loss = 0.0058
Batch 24: Loss = 0.0122
Batch 25: Loss = 0.0438
Batch 26: Loss = 0.0327
Batch 27: Loss = 0.0271
Batch 28: Loss = 0.0505
Batch 29: Loss = 0.0287
Batch 30: Loss = 0.0556
Batch 31: Loss = 0.0584
Batch 32: Loss = 0.0355
Batch 33: Loss = 0.0428
Batch 34: Loss = 0.0390
Batch 35: Loss = 0.0976

Epoch 2/5
Batch 1: Loss = 0.6431
Batch 2: Loss = 0.5567
Batch 3: Loss = 0.4177
Batch 4: Loss = 0.1248
Batch 5: Loss = 0.1845
Batch 6: Loss = 0.0881
Batch 7: 

### Model Evaluation

In [19]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# After training rounds are completed...

print("\n--- Evaluating Federated Model ---")

# Prepare test data (hidden states from clients)
client_hidden_test = []

for client_id, (client_name, data) in enumerate(client_data.items()):
    X_test = data['X_test']

    # Reshape input for LSTM if necessary
    #X_test_reshaped = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

    #hidden_out = client_models[client_name].predict(X_test_reshaped, verbose=1)
    hidden_out = client_models[client_name].predict(X_test, verbose=1)
    client_hidden_test.append(hidden_out)

# Stack client outputs together (axis=-1 to match concatenation during training)
server_input_test = np.concatenate(client_hidden_test, axis=-1)

# Make final predictions
y_pred = server_model.predict(server_input_test, verbose=1).flatten()

# Assuming all clients share the same y_test (if not, pick one client's y_test)
sample_client = next(iter(client_data.values()))
y_true = sample_client['y_test']

# Compute evaluation metrics
mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mse)

print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")


--- Evaluating Federated Model ---
7/7 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
Test MSE: 0.0106
Test MAE: 0.0833
Test RMSE: 0.1029


In [20]:
# End of code 4/22/25